# 6.37: Tiny Aya Global with a search tool over the earlier chat

This notebook shows the shipped LoRA adapter (6.37, seed 82) on the project's scripted demo chat, beside the
untrained model, in the harness it was tested in: the last three exchanges in view, everything older reachable
only through a full-text search tool.

- **Part A** (CPU, no login, under a minute) prints the two recorded runs, every turn, from the repository.
- **Part B** (a GPU runtime; about 15 minutes on a free Colab T4) reruns both models live. Tiny Aya is gated, so
  Part B needs a Hugging Face account that has accepted its licence.

Four questions are an illustration, not a measurement. The measurements, and the limits, are in the
[README](https://github.com/Vee-industries/aya_retrieval#readme).

## Part A: the recorded runs

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/aya_retrieval") if Path("/content").exists() else Path("aya_retrieval")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/Vee-industries/aya_retrieval", str(REPO)], check=True
    )
print(
    subprocess.run(
        ["git", "-C", str(REPO), "log", "-1", "--format=repository commit %h, %ad"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)

In [ ]:
import json


def show(path):
    run = json.loads(Path(path).read_text(encoding="utf-8"))
    p = run["provenance"]
    print(
        f"{Path(path).name}: commit {p['git_commit'][:8]}, {p['utc_time']},",
        "torch",
        p["packages"]["torch"],
    )
    return run["turns"]


tuned = show(REPO / "results/demo/fix17cs37_82.json")
base = show(REPO / "results/demo/base_r175.json")
for n, (t, b) in enumerate(zip(tuned, base, strict=True), 1):
    print(f"\n{n}. you: {t['message']}")
    for name, turn in (("6.37", t), ("base", b)):
        searches = "".join(f" [search: {q}]" for q in turn["queries"])
        print(f"   {name}:{searches} {turn['reply']}")

The grades for the four questions, and the earlier adapters' replies on the two 6.37 gets wrong, are in
[`docs/demo.md`](https://github.com/Vee-industries/aya_retrieval/blob/main/docs/demo.md).

## Part B: run it live

1. Choose a GPU runtime (Runtime, Change runtime type, T4 GPU).
2. Accept the licence on [CohereLabs/tiny-aya-global](https://huggingface.co/CohereLabs/tiny-aya-global).
3. Give the notebook a Hugging Face token: either add it as the Colab secret `HF_TOKEN` with notebook access, or
   run the login cell below. A fine-grained token needs "Read access to contents of all public gated repos you can
   access" ticked.

In [ ]:
import sys

import torch

assert sys.version_info >= (3, 12), "AgentTune needs Python 3.12 or later"
assert torch.cuda.is_available(), "Part B needs a GPU runtime"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# The toolkit at the commit every result was made with. Colab's own torch is kept.
VENDOR = REPO.parent / "vendor"
AGENTTUNE = VENDOR / "AgentTune"
if not AGENTTUNE.exists():
    AGENTTUNE_URL = "https://github.com/Lexsi-Labs/AgentTune"
    subprocess.run(["git", "clone", AGENTTUNE_URL, str(AGENTTUNE)], check=True)
subprocess.run(
    ["git", "-C", str(AGENTTUNE), "checkout", "-q", "d4db4c12d52342227c4113a70183d0f64aa2f345"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(AGENTTUNE)], check=True
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.22.1",
        "peft==0.21.2",
        "accelerate==1.15.0",
        "safetensors==0.8.0",
        "huggingface_hub==1.33.0",
        "datasets==5.1.0",
        "langchain-text-splitters==1.1.3",
        "psutil",
        "pyyaml",
    ],
    check=True,
)
sys.path.insert(0, str(REPO / "src"))

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login

    notebook_login()

In [ ]:
import hashlib

from huggingface_hub import snapshot_download

from aya_retrieval.models import MODEL_ID, MODEL_REVISION

snapshot_download(MODEL_ID, revision=MODEL_REVISION)
ADAPTER = Path(
    snapshot_download(
        "KevinVaillancourt/aya-retrieval-6.37", revision="47a0ce601971460e8a5d8c196fdc20f5a47e4631"
    )
)
manifest = json.loads((REPO / "results/freeze/sealed_manifest.json").read_text(encoding="utf-8"))
frozen = manifest["frozen"]["model"]["adapter_files"]["adapter_model.safetensors"]
got = hashlib.sha256((ADAPTER / "adapter_model.safetensors").read_bytes()).hexdigest()
assert got == frozen, "adapter weights differ from the frozen model"
print("adapter_model.safetensors matches the sealed manifest")
# The published adapter_config.json differs from the frozen one only in base_model_name_or_path
# (a local cache path, set to the Hub id) and revision, as recorded at publication (R-172).
config = json.loads((ADAPTER / "adapter_config.json").read_text(encoding="utf-8"))
print("adapter_config:", config["base_model_name_or_path"], config["revision"])

In [ ]:
from peft import PeftModel

from aya_retrieval.chat import MAX_TOKENS, Chat
from aya_retrieval.inference import GREEDY, generate_from_messages
from aya_retrieval.models import load_model

model, tokenizer = load_model()  # bfloat16, as every result was made
print(model.dtype, next(model.parameters()).device)


RUNS = REPO.parent / "notebook_runs"  # search indexes; outside the repository
RUNS.mkdir(exist_ok=True)


def run(model, name):
    def generate(messages):
        return generate_from_messages(model, tokenizer, messages, GREEDY, MAX_TOKENS)

    index = RUNS / f"{name}.sqlite"
    index.unlink(missing_ok=True)
    chat = Chat(generate, index)
    turns = []
    for message in (REPO / "demo/chat.txt").read_text(encoding="utf-8").splitlines():
        if message:
            result = chat.send(message)
            turns.append({"message": message, "queries": result.queries, "reply": result.reply})
    return turns


live_base = run(model, "base")  # before the adapter wraps the model
model = PeftModel.from_pretrained(model, str(ADAPTER))
model.eval()
live_tuned = run(model, "tuned")

In [ ]:
# A different GPU, dtype path or torch version can change a greedy reply;
# any turn that differs from the recorded run is marked.
for n, (lt, lb, t, b) in enumerate(zip(live_tuned, live_base, tuned, base, strict=True), 1):
    print(f"\n{n}. you: {lt['message']}")
    for name, live, rec in (("6.37", lt, t), ("base", lb, b)):
        searches = "".join(f" [search: {q}]" for q in live["queries"])
        same = live["queries"] == rec["queries"] and live["reply"] == rec["reply"]
        mark = "" if same else "   <- differs from the recorded run"
        print(f"   {name}:{searches} {live['reply']}{mark}")

## Try your own messages

`Chat` keeps the last three exchanges in view and indexes everything older for the search tool. With the
adapter on, `model` is 6.37; inside `with model.disable_adapter():` it is base.

In [ ]:
def mine_generate(messages):
    return generate_from_messages(model, tokenizer, messages, GREEDY, MAX_TOKENS)


mine = Chat(mine_generate, RUNS / "mine.sqlite")
result = mine.send("my cat is called Pebble and she is nine")
print(result.queries, result.reply)